# Лабораторная работа 3. Знакомство с "aha moment"

Работа основана на коде статьи `Mini-R1: Reproduce Deepseek R1 „aha moment“ a RL tutorial` ([ссылка](https://huggingface.co/blog/open-r1/mini-r1-contdown-game)) автора Philipp Schmid, которая посвящена воспроизведению процесса обучения рассуждениям модели deepseek-R1 на задаче из игры "Countdown" (используется датасет [ссылка](https://huggingface.co/datasets/Jiayi-Pan/Countdown-Tasks-3to4)).

In [ ]:
pip install trl

Будем использовать GRPO политику для обучения с подкреплением.

**Задание**. Запустите код и разберите статью. Разберитесь какой задаче обучается модель.

Ответьте **письменно** на следующие вопросы:
1. Какой формат имеют входные данные, которые подаются модели? Приведите пример.
2. Какие используются reward-функции?
3. Какая используется функция потерь?
4. Что такое "aha moment"?
5. Как в коде происходит запись сгенерированного ответа в лог-файл `completion_samples.txt`?

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset
import re
from trl import GRPOConfig, GRPOTrainer, get_peft_config, ModelConfig
import logging
from logging import FileHandler, StreamHandler
import random
import os

In [ ]:
logger = logging.getLogger("app")
logger.setLevel(logging.DEBUG)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
fh = FileHandler("app.log", encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(fmt)
sh = StreamHandler()
sh.setLevel(logging.INFO)
sh.setFormatter(fmt)
logger.addHandler(fh)
logger.addHandler(sh)

In [ ]:
def format_reward_func(completions, target, **kwargs):
    """
    Format: <think>...</think><answer>...</answer>
    Args:
        completions (list[str]): Generated outputs
        target (list[str]): Expected answers

      Returns:
          list[float]: Reward scores
    """
    rewards = []

    for completion, _ in zip(completions, target):

      try:
        completion = "<think>" + completion
        if random.random() < 0.1:
          os.makedirs("completion_samples", exist_ok=True)
          log_file = os.path.join("completion_samples", "completion_samples.txt")
          with open(log_file, "a") as f:
            f.write(f"\n\n==============\n")
            f.write(completion)
        regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\/think>\n<answer>([\s\S]*?)<\/answer>$"

        match = re.search(regex, completion, re.DOTALL)
        if match is None or len(match.groups()) != 2:
            rewards.append(0.0)
        else:
            rewards.append(1.0)
      except Exception:
        rewards.append(0.0)
    return rewards

In [ ]:
def equation_reward_func(completions, target, nums, **kwargs):
    """
    Evaluates completions based on:
    2. Mathematical correctness of the answer

    Args:
        completions (list[str]): Generated outputs
        target (list[str]): Expected answers
        nums (list[str]): Available numbers

    Returns:
        list[float]: Reward scores
    """
    rewards = []
    for completion, gt, numbers in zip(completions, target, nums):
      try:
        completion = "<think>" + completion
        match = re.search(r"<answer>(.*?)<\/answer>", completion)
        if match is None:
            rewards.append(0.0)
            continue
        equation = match.group(1).strip()
        used_numbers = [int(n) for n in re.findall(r'\d+', equation)]

        if sorted(used_numbers) != sorted(numbers):
            rewards.append(0.0)
            continue
        allowed_pattern = r'^[\d+\-*/().\s]+$'
        if not re.match(allowed_pattern, equation):
           rewards.append(0.0)
           continue

        result = eval(equation, {"__builtins__": None}, {})
        if abs(float(result) - float(gt)) < 1e-5:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
      except Exception:
            rewards.append(0.0)
    return rewards

In [ ]:
def generate_r1_prompt(numbers, target):
    r1_prefix = [{
        "role": "system",
        "content": "You are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer."
      },
      {
        "role": "user",
        "content": f"Using the numbers {numbers}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>."
      },
      {
        "role": "assistant",
        "content": "Let me solve this step by step.\n<think>"
      }]
    return {"prompt": tokenizer.apply_chat_template(r1_prefix, tokenize=False, continue_final_message=True)}

In [ ]:
dataset_id = "Jiayi-Pan/Countdown-Tasks-3to4"
dataset = load_dataset(dataset_id, split="train")
dataset = dataset.shuffle(seed=42).select(range(50000))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

In [ ]:
dataset = dataset.map(lambda x: generate_r1_prompt(x["nums"], x["target"]))

train_test_split = dataset.train_test_split(test_size=0.1)

train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]
logger.info(f"SAMPLE OF TRAIN_DATASET:\n{train_dataset[0]}")

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

2025-12-14 13:10:29,032 | INFO | app | SAMPLE OF TRAIN_DATASET:
{'target': 28, 'nums': [94, 33, 89], 'prompt': '<|im_start|>system\nYou are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer.<|im_end|>\n<|im_start|>user\nUsing the numbers [94, 33, 89], create an equation that equals 28. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.<|im_end|>\n<|im_start|>assistant\nLet me solve this step by step.\n<think>'}
INFO:app:SAMPLE OF TRAIN_DATASET:
{'target': 28, 'nums': [94, 33, 89], 'prompt': '<|im_start|>system\nYou are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer.<|im_end|>\n<|im_start|>user\nUsing the numbers [94, 33, 89], create an equation tha

In [ ]:
logger.info("LOAD MODEL")
model_config = ModelConfig(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="bfloat16",
    attn_implementation="flash_attention_2",
    use_peft=True,
    load_in_4bit=True,
)

2025-12-14 14:02:07,315 | INFO | app | LOAD MODEL
2025-12-14 14:02:07,315 | INFO | app | LOAD MODEL
INFO:app:LOAD MODEL
<string>:24: FutureWarning: `torch_dtype` is deprecated and will be removed in version 0.27.0, please use `dtype` instead.


In [ ]:
logger.info("INITIALIZING PARAMETERS")
training_args = GRPOConfig(
    output_dir="qwen-r1-aha-moment",
    learning_rate=5e-7,
    lr_scheduler_type="cosine",
    logging_steps=10,
    max_steps=100,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,
    # GRPO specific parameters
    max_prompt_length=256,
    max_completion_length=1024,
    num_generations=2,
    beta=0.001,
    report_to="none"

)

2025-12-14 14:02:09,010 | INFO | app | INITIALIZING PARAMETERS
2025-12-14 14:02:09,010 | INFO | app | INITIALIZING PARAMETERS
INFO:app:INITIALIZING PARAMETERS
<string>:192: FutureWarning: The `max_prompt_length` argument is deprecated and will be removed in version 0.28.0. You should instead filter your dataset before training to ensure that prompts do not exceed your desired length.


In [ ]:
trainer = GRPOTrainer(
    model=model_config.model_name_or_path,
    reward_funcs=[format_reward_func, equation_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=get_peft_config(model_config),
)

logger.info("START TRAINING")
trainer.train()

logger.info("END TRAINING")
logger.info("SAVE TRAINED MODEL")
trainer.save_model(training_args.output_dir)

The model is already on multiple devices. Skipping the move to device specified in `args`.
2025-12-14 14:02:17,345 | INFO | app | START TRAINING
2025-12-14 14:02:17,345 | INFO | app | START TRAINING
INFO:app:START TRAINING
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.000000
20,0.000000
30,-0.007400
40,0.024700
50,0.040300
60,-0.023600
70,0.061800
80,0.000000
90,0.000000
100,0.050800


2025-12-14 14:26:56,271 | INFO | app | END TRAINING
2025-12-14 14:26:56,271 | INFO | app | END TRAINING
INFO:app:END TRAINING
2025-12-14 14:26:56,274 | INFO | app | SAVE TRAINED MODEL
2025-12-14 14:26:56,274 | INFO | app | SAVE TRAINED MODEL
INFO:app:SAVE TRAINED MODEL
